In [1]:
import pandas as pd
import numpy as np
import re
import difflib
import importlib
import cleaning as cln
importlib.reload(cln)

<module 'cleaning' from 'd:\\DS108\\DS108_Phone-Reccomendation\\cleaning.py'>

In [2]:
cps = pd.read_csv('cellphones_raw.csv')
cps = cps.drop(columns=['Unnamed', ' 0', 'Link'])
att = pd.read_csv(r'antutu_socket.csv')

In [3]:
gsm = pd.read_csv(r'all_phones_final.csv')
mem = gsm[['name_clean', 'Memory | Internal']]
ref_rate = gsm[['name_clean', 'Display | Type']]
battery = gsm[['name_clean', 'Battery | Type']]

In [4]:
thresh_limit = 0.5 * cps.shape[0]

cols_to_check = [col for col in cps.columns if col != 'Price' and col != "Tần số quét"]

cols_to_keep = (
    cps[cols_to_check].dropna(thresh=thresh_limit, axis=1).columns.tolist()
)

cols_to_keep.append("Price")
cols_to_keep.append("Tần số quét")

cps = cps[cols_to_keep]

In [5]:
feature_mapping = {
    "Tên": "Name",
    "Price": "Price",
    "Kích thước màn hình": "Screen Size",
    "Công nghệ màn hình": "Display",
    "Camera sau": "Rear Camera",
    "Camera trước": "Front Camera",
    "Chipset": "Chipset",
    "Công nghệ NFC": "NFC",
    "Bộ nhớ trong": "ROM",
    "Thẻ SIM": "SIM Card",
    "Hệ điều hành": "Operating System",
    "Độ phân giải màn hình": "Screen Resolution",
    "Tính năng màn hình": "Display Features",
    "Loại CPU": "CPU",
    "Dung lượng RAM": "RAM",
    "Pin": "Battery",
}

cps = cps.rename(columns=feature_mapping)


In [6]:
cps.columns

Index(['Name', 'Screen Size', 'Display', 'Rear Camera', 'Front Camera',
       'Chipset', 'NFC', 'ROM', 'SIM Card', 'Operating System',
       'Screen Resolution', 'Display Features', 'CPU', 'RAM', 'Battery',
       'Price', 'Tần số quét'],
      dtype='str')

In [7]:
cps["Name"] = cps["Name"].apply(cln.clean_phone_name)
mem['name_clean'] = mem['name_clean'].apply(cln.clean_phone_name)
ref_rate['name_clean'] = ref_rate['name_clean'].apply(cln.clean_phone_name)
battery['name_clean'] = battery['name_clean'].apply(cln.clean_phone_name)

CLEANING AND EXTRACT

In [ ]:
#NFC
cps['NFC'] = cps['NFC'].map(lambda x : 1 if x == "Có" else 0)

#Brand
cps['Brand'] = cps['Name'].apply(cln.get_brand)

#OS
cps[["OS_Name", "OS_Version"]] = cps.apply(
    lambda row: pd.Series(
        cln.advanced_clean_os(
            row["Operating System"],
            row["Brand"]
        )
    ),
    axis=1
)


#Chipset
cps['Chipset'] = cps['Chipset'].apply(cln.clean_chipset)
cps = cln.add_chipset_info(att, cps)
cps['Chipset_name'] = cps['Chipset'].apply(cln.extract_chipset)
cps['Chipset_gen'] = cps['Chipset'].apply(cln.extract_chipset_gen)

#gpu
cps['gpu_name'] = cps['gpu'].apply(cln.extract_gpu)
cps['gpu_gen'] = cps['gpu'].apply(cln.extract_gpu_gen)

#Screen Size, clock
cols_to_clean = ["Screen Size", "clock"]
for col in cols_to_clean:
    cps[col] = cps[col].apply(cln.clean_metrics)

#Refresh Rate 
cps['Refresh Rate'] = cps.apply(
    lambda row: cln.extract_refresh_rate(row['Tần số quét'], row['Display Features']),
    axis=1
)
ref_rate["Refresh Rate"] = ref_rate["Display | Type"].apply(cln.extract_hz_gsm)
ref_rate['name_clean'] = ref_rate['name_clean'].apply(cln.remove_first_word_if_iphone)
ref_rate.drop(columns= 'Display | Type', inplace=True)
ref_rate_clean = ref_rate[['name_clean', 'Refresh Rate']].drop_duplicates(subset=['name_clean'])

cps_merged = pd.merge(
    cps,
    ref_rate_clean[["name_clean", "Refresh Rate"]],  
    left_on="Name",
    right_on="name_clean",
    how="left",
    suffixes=("", "_from_ref"),  
)

cps_merged["Refresh Rate"] = cps_merged["Refresh Rate"].fillna(
    cps_merged["Refresh Rate_from_ref"]
)

cps_merged.drop(columns=["name_clean", "Refresh Rate_from_ref"], inplace=True)

cps = cps_merged

#RAM and ROM
cps = cln.add_ram(mem, cps)
cps["RAM"] = cps["RAM"].apply(cln.clean_storage)
cps["ROM"] = cps["ROM"].apply(cln.clean_storage)

#Battery
cps['Battery'] = cps['Battery'].apply(cln.clean_battery)
cps = cln.fill_iphone_battery(cps)
battery["Battery"] = battery["Battery | Type"].apply(cln.extract_battery_gsm)
battery.drop(columns= 'Battery | Type', inplace=True)
battery_clean = battery[['name_clean', 'Battery']].drop_duplicates(subset=['name_clean'])

cps_merged = pd.merge(
    cps,
    battery_clean[["name_clean", "Battery"]],  
    left_on="Name",
    right_on="name_clean",
    how="left",
    suffixes=("", "_from_ref"),  
)

cps_merged["Battery"] = cps_merged["Battery"].fillna(
    cps_merged["Battery_from_ref"]
)

cps_merged.drop(columns=["name_clean", "Battery_from_ref"], inplace=True)

cps = cps_merged

#CPU
cps[["total_cores", "max_freq_ghz", "min_freq_ghz", "weighted_mean"]] = cps["architecture"].apply(
    lambda x: pd.Series(cln.extract_architecture(x))
)

#Resolution
cps["Reso_Width"], cps["Reso_Height"] = zip(
    *cps["Screen Resolution"].apply(cln.extract_res_row)
)

#SIM
(
    cps["Nano_SIM_Count"],
    cps["eSIM_Count"],
    cps["Micro_SIM_Count"],
    cps["Mini_SIM_Count"],
) = zip(*cps["SIM Card"].apply(cln.clean_sim_options))

#Camera
cps = cln.extract_camera_info(cps)

#Display
cps['Display'] = cps['Display'].apply(cln.extract_display_type)



DERIVING

In [9]:
#PPI
median_screen = cps.loc[cps['Screen Size'] > 0, 'Screen Size'].median()
cps['Screen Size'] = cps['Screen Size'].replace(0, median_screen)

cps['PPI'] = (
    np.sqrt(cps['Reso_Width']**2 + cps['Reso_Height']**2) / cps['Screen Size']
).round(1)

cps.drop(columns=['Reso_Width', 'Reso_Height'], inplace=True)

#SIM
cps['SIM_total'] = (
    cps['Nano_SIM_Count'] +
    cps['eSIM_Count'] +
    cps['Micro_SIM_Count'] +
    cps['Mini_SIM_Count']
).clip(upper=2)
cps['has_eSIM'] = (cps['eSIM_Count'] > 0).astype(int)

cps.drop(columns=[ 'Nano_SIM_Count', 'Micro_SIM_Count', 'Mini_SIM_Count', 'eSIM_Count'], inplace=True)



DROPPING

In [10]:
drop = [
    'Rear Camera', 'Front Camera', 'Screen Resolution', 'Display Features',
    'Operating System', 'SIM Card', 'OS_Is_Android', 'Tần số quét',
    'CPU', 'architecture', 'camera_score']

cps.drop(columns=drop, inplace = True, errors='ignore')

In [11]:
mask_not_smartphone = (
    (cps['Screen Size'] < 4.0) |
    (cps['Name'].str.lower().str.contains('tab|pad|win rt', na=False))
)

not_smartphone = cps[mask_not_smartphone]

cps = cps[~mask_not_smartphone].reset_index(drop=True)
print(f'Đã drop {mask_not_smartphone.sum()} sản phẩm, còn lại {len(cps)}')
print(not_smartphone[['Name', 'Screen Size']].head(5))  # xem 20 dòng đầu

Đã drop 42 sản phẩm, còn lại 924
                      Name  Screen Size
149         masstel izi 15         1.77
161  trẻ em masstel alfa 5         3.50
164             nokia 3210         2.40
165          nokia hmd 105         2.40
169              nokia 220         2.80


In [12]:
cps['Price'] = cps['Price'].apply(cln.clean_price)
with_price = cps[cps['Price'].notna()]
without_price = cps[cps['Price'].isna()]
print(len(with_price))
print(len(without_price))

317
607


In [13]:
with_price['Brand'].value_counts()

Brand
samsung    76
apple      49
xiaomi     43
oppo       41
nubia      24
honor      18
tecno      17
meizu       7
itel        5
zte         5
huawei      5
realme      5
nothing     4
vivo        4
sony        3
vsmart      3
poco        2
asus        2
nokia       2
benco       1
masstel     1
Name: count, dtype: int64

In [14]:
without_price['Brand'].value_counts()

Brand
xiaomi       114
vivo          98
oppo          59
samsung       48
realme        45
honor         45
huawei        30
tecno         27
oneplus       22
nubia         21
google        20
infinix       10
motorola      10
nokia          8
asus           5
masstel        5
sony           4
nothing        4
zte            4
bphone         3
htc            3
doogee         3
apple          2
poco           2
sharp          2
meizu          2
leitz          2
iqoo           2
blackview      2
lenovo         2
tcl            2
itel           1
Name: count, dtype: int64

In [15]:
brand_counts_raw = with_price["Brand"].value_counts()

# Tìm các brand có từ 10 sản phẩm trở lên trong tập có giá
valid_brands = brand_counts_raw[brand_counts_raw >= 5].index.tolist()
removed_brands = brand_counts_raw[brand_counts_raw < 5].index.tolist()

print(f"Các hãng ĐƯỢC GIỮ LẠI (>= 5 máy trong tập gốc): {valid_brands}")
print(f"Các hãng BỊ LOẠI BỎ (< 5 máy trong tập gốc): {removed_brands}")

# Tạo tập train sạch sau khi lọc brand
with_price = with_price[with_price["Brand"].isin(valid_brands)].copy()
print(
    f"-> Kích thước tập Train SẠCH (đã có giá): {len(with_price)} dòng (Giảm từ {len(with_price)})"
)

Các hãng ĐƯỢC GIỮ LẠI (>= 5 máy trong tập gốc): ['samsung', 'apple', 'xiaomi', 'oppo', 'nubia', 'honor', 'tecno', 'meizu', 'itel', 'zte', 'huawei', 'realme']
Các hãng BỊ LOẠI BỎ (< 5 máy trong tập gốc): ['nothing', 'vivo', 'sony', 'vsmart', 'poco', 'asus', 'nokia', 'benco', 'masstel']
-> Kích thước tập Train SẠCH (đã có giá): 295 dòng (Giảm từ 295)


LẤY THÊM CÁC ĐIỆN THOẠI CHƯA CÓ GIÁ

In [16]:
no_price_with_antutu = without_price[
    (without_price["Brand"].isin(valid_brands)) & (without_price["antutu_11"].notna())
].copy()

print(
    f"-> Tổng số máy chưa có giá, HỢP LỆ thương hiệu và CÓ AnTuTu: {len(no_price_with_antutu)}"
)


# =====================================================================
# BƯỚC 3: TIẾN HÀNH LẤY MẪU 2 LỚP (SAMPLING) TRÊN POOL ĐÃ LỌC
# =====================================================================
TOTAL_TARGET = 300
print("\n--- BƯỚC 3: TIẾN HÀNH SAMPLING ---")

sampled_no_price = pd.DataFrame()
remaining_slots = TOTAL_TARGET

# --- LỚP 1 — PROPORTIONAL SAMPLING ---
print("\n[LỚP 1: PROPORTIONAL SAMPLING]")
# Tính lại tỷ lệ phần trăm các brand dựa trên tập train SẠCH mới
brand_props_train = with_price["Brand"].value_counts() / len(with_price)

for brand, prop in brand_props_train.items():
    # Tính quota lý thuyết dựa trên tỷ lệ mới
    quota = int(round(prop * TOTAL_TARGET))

    # Lấy số lượng khả dụng trong pool đã lọc sạch ở Bước 2
    brand_pool = no_price_with_antutu[no_price_with_antutu["Brand"] == brand]
    available = len(brand_pool)

    # Nguyên tắc: actual = min(quota, available)
    actual = min(quota, available)

    if actual > 0:
        sampled_brand = brand_pool.sample(n=actual, random_state=42)
        sampled_no_price = pd.concat([sampled_no_price, sampled_brand])
        # Xóa các dòng đã chọn khỏi pool cuốn chiếu
        no_price_with_antutu = no_price_with_antutu.drop(sampled_brand.index)

    remaining_slots -= actual
    print(
        f"Brand: {brand:<12} | Quota: {quota:<3} | Có sẵn: {available:<3} | Thực lấy: {actual:<3}"
    )

print(f"-> Số slot còn dư sau Lớp 1 chuyển vào quỹ chung: {remaining_slots}")

# --- LỚP 2 — BOOST SAMPLING ---
print("\n[LỚP 2: BOOST SAMPLING]")
# Cấu hình boost (chỉ giữ lại các hãng hợp lệ nằm trong valid_brands)
boost_config_raw = {
    "Vivo": 30,
    "Realme": 15,
    "Huawei": 10,
    "OnePlus": 15,
    "Google": 12,
}
boost_config = {k: v for k, v in boost_config_raw.items() if k.lower() in [b.lower() for b in valid_brands]}

for brand, boost_quota in boost_config.items():
    if remaining_slots <= 0:
        break

    # Tìm kiếm không phân biệt hoa thường để tránh lỗi dữ liệu
    brand_pool = no_price_with_antutu[
        no_price_with_antutu["Brand"].str.lower() == brand.lower()
    ]
    available = len(brand_pool)
    actual = min(boost_quota, available, remaining_slots)

    if actual > 0:
        sampled_brand = brand_pool.sample(n=actual, random_state=42)
        sampled_no_price = pd.concat([sampled_no_price, sampled_brand])
        no_price_with_antutu = no_price_with_antutu.drop(sampled_brand.index)
        remaining_slots -= actual
        print(
            f"Boost Brand: {brand:<8} | Yêu cầu: {boost_quota:<2} | Có sẵn: {available:<3} | Thực lấy: {actual:<2}"
        )

# --- ĐIỀN NỐT CÁC SLOT DƯ CUỐI CÙNG ---
if remaining_slots > 0 and len(no_price_with_antutu) > 0:
    actual = min(remaining_slots, len(no_price_with_antutu))
    final_fill = no_price_with_antutu.sample(n=actual, random_state=42)
    sampled_no_price = pd.concat([sampled_no_price, final_fill])
    remaining_slots -= actual
    print(
        f"\n[PHÒNG THỦ] Điền nốt {actual} slot còn lại từ pool tổng để đủ target 300 dòng."
    )

# =====================================================================
# THỐNG KÊ KẾT QUẢ CUỐI CÙNG
# =====================================================================
print("\n" + "=" * 50)
print(f"TỔNG SỐ DÒNG CHƯA CÓ GIÁ ĐÃ LẤY THÀNH CÔNG: {len(sampled_no_price)}")
print("=" * 50)
print("Thống kê số lượng theo từng Brand trong tập dữ liệu mới:")
print(sampled_no_price["Brand"].value_counts())

-> Tổng số máy chưa có giá, HỢP LỆ thương hiệu và CÓ AnTuTu: 315

--- BƯỚC 3: TIẾN HÀNH SAMPLING ---

[LỚP 1: PROPORTIONAL SAMPLING]
Brand: samsung      | Quota: 77  | Có sẵn: 45  | Thực lấy: 45 
Brand: apple        | Quota: 50  | Có sẵn: 2   | Thực lấy: 2  
Brand: xiaomi       | Quota: 44  | Có sẵn: 85  | Thực lấy: 44 
Brand: oppo         | Quota: 42  | Có sẵn: 46  | Thực lấy: 42 
Brand: nubia        | Quota: 24  | Có sẵn: 14  | Thực lấy: 14 
Brand: honor        | Quota: 18  | Có sẵn: 36  | Thực lấy: 18 
Brand: tecno        | Quota: 17  | Có sẵn: 19  | Thực lấy: 17 
Brand: meizu        | Quota: 7   | Có sẵn: 2   | Thực lấy: 2  
Brand: itel         | Quota: 5   | Có sẵn: 0   | Thực lấy: 0  
Brand: zte          | Quota: 5   | Có sẵn: 2   | Thực lấy: 2  
Brand: huawei       | Quota: 5   | Có sẵn: 27  | Thực lấy: 5  
Brand: realme       | Quota: 5   | Có sẵn: 37  | Thực lấy: 5  
-> Số slot còn dư sau Lớp 1 chuyển vào quỹ chung: 104

[LỚP 2: BOOST SAMPLING]
Boost Brand: Realme   | Yêu cầu:

Đánh dấu nguồn gốc của cột Price, các dòng có price lấy từ cellphoneS là 1, các dòng dùng predict để fill là 0

In [17]:
with_price['Price_source'] = 1
sampled_no_price['Price_source'] = 0

In [18]:
df = pd.concat([sampled_no_price, with_price])

In [19]:
df = cln.fill_missing_os_by_brand(
    df, brand_col="Brand", os_col="OS_Version"
)

df = cln.fill_camera_by_chipset(df)

df = cln.fill_ppi(df)

df = cln.fill_refresh_rate(df)



In [20]:
# Define the core columns (deduplicated to avoid double-counting)
core_cols = ['Screen Size', 'Battery', 'RAM', 'ROM']

# Compute the null count per row
df['core_null_count'] = df[core_cols].isna().sum(axis=1)

tier_2 = df.query('core_null_count > 3')

df_filled = cln.fill_ram_by_chipset(df)
df_filled = cln.fill_rom_by_chipset(df_filled)
df_filled = cln.fill_battery_by_brand(df_filled)
df_filled = cln.fill_screen_size_by_brand_chipset(df_filled)

tier_2 = df_filled.loc[tier_2.index]

tier_1 = df.query('core_null_count <= 3')

display_battery_cols = ['Screen Size', 'PPI', 'Refresh Rate', 'Battery']
a = cln.impute_numeric_with_knn(df, numeric_cols=display_battery_cols, n_neighbors=5)
tier_1 = a.loc[tier_1.index]

hardware_cols = ['RAM', 'ROM', 'antutu_11']
a = cln.impute_numeric_with_knn(df, numeric_cols=hardware_cols, n_neighbors=5)
tier_1 = a.loc[tier_1.index]

camera_cols = ['rear_count', 'rear_mp_max', 'front_mp', 'rear_ois', 'rear_telephoto', 'rear_wide', 'front_f/', 'rear_f/']
a = cln.impute_numeric_with_knn(df, numeric_cols=camera_cols, n_neighbors=5)
tier_1 = a.loc[tier_1.index]

df = pd.concat([tier_1, tier_2], axis=0)
df = df.reset_index(drop=True)

df = cln.fill_battery_by_brand(df)
df = cln.fill_screen_size_by_brand_chipset(df)
df = cln.fill_rom_by_chipset(df)
df = cln.fill_ram_by_chipset(df)

In [21]:
# 1. Định nghĩa các nhóm thuộc tính
median_features = [
    "antutu_11",
    "clock",
    "total_cores",
    "max_freq_ghz",
    "min_freq_ghz",
    "weighted_mean"
]  # Nhóm trung vị
mode_features = ["gpu"]  # Nhóm yếu vị

# 2. Thực hiện gọi hàm xử lý đồng loạt
df = cln.fill_missing_by_chipset(
    df, median_cols=median_features, mode_cols=mode_features
)

# 3. Kiểm tra lại kết quả xem còn sót dòng trống nào không
all_cols_to_check = median_features + mode_features
print("Số lượng dòng Null còn sót lại sau khi xử lý:")
print(df[all_cols_to_check].isna().sum())

Số lượng dòng Null còn sót lại sau khi xử lý:
antutu_11        0
clock            0
total_cores      0
max_freq_ghz     0
min_freq_ghz     0
weighted_mean    0
gpu              0
dtype: int64


In [22]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

# ==========================================
# BƯỚC SỬA LỖI TRƯỚC KHI CHIA TẬP: VÁ CỘT CHIPSET GỐC
# ==========================================
# Đề phòng trường hợp cột Chipset gốc bị null, tự động vá bằng cách ghép Chipset_name và Chipset_gen
null_chipset_mask = df["Chipset"].isna()
if null_chipset_mask.sum() > 0:
    df.loc[null_chipset_mask, "Chipset"] = (
        df.loc[null_chipset_mask, "Chipset_name"].astype(str)
        + " "
        + df.loc[null_chipset_mask, "Chipset_gen"].astype(str)
    )
    print(
        f"-> Đã vá {null_chipset_mask.sum()} dòng Chipset trống dựa trên Name và Gen."
    )

# ==========================================
# BƯỚC 1 — TÁCH TRAIN VÀ PREDICT SET
# ==========================================
train_df = df[df["Price"].notna()].copy()
predict_df = df[df["Price"].isna()].copy()

print(f"Kích thước tập Train (Có giá): {train_df.shape[0]} dòng")
print(f"Kích thước tập Predict (Cần điền giá): {predict_df.shape[0]} dòng")

# ==========================================
# BƯỚC 2 — XÁC ĐỊNH FEATURE COLUMNS
# ==========================================
# Danh sách 16 features chuẩn hóa dạng số đưa vào mô hình AI
feature_cols = [
    "RAM",
    "ROM",
    "Battery",
    "Screen Size",
    "Refresh Rate",
    "antutu_11",
    "clock",
    "total_cores",
    "max_freq_ghz",
    "min_freq_ghz",
    "Brand_enc",
    "Chipset_enc",
    "Chipset_name_enc",
    "Chipset_gen_enc",
    "gpu_name_enc",
    "gpu_gen_enc",
]

# ==========================================
# BƯỚC 3 — ENCODE TẤT CẢ CATEGORICAL FEATURES (STRING)
# ==========================================
# Định nghĩa các cặp cột Object/String cần mã hóa
categorical_mappings = {
    "Brand": "Brand_enc",
    "Chipset": "Chipset_enc",
    "Chipset_name": "Chipset_name_enc",
    "Chipset_gen": "Chipset_gen_enc",
    "gpu_name": "gpu_name_enc",
    "gpu_gen": "gpu_gen_enc",
}

# Điền giá trị trống mặc định cho các cột chữ và ép kiểu string
for src_col in categorical_mappings.keys():
    df[src_col] = df[src_col].fillna("Unknown").astype(str)

# Thực hiện Label Encoding trên TOÀN BỘ tập dữ liệu df để tránh lỗi nhãn lạ
for src_col, enc_col in categorical_mappings.items():
    le = LabelEncoder()
    df[enc_col] = le.fit_transform(df[src_col])

# Đồng bộ hóa các cột số mã hóa mới tạo sang hai tập train_df và predict_df
for enc_col in categorical_mappings.values():
    train_df[enc_col] = df.loc[train_df.index, enc_col]
    predict_df[enc_col] = df.loc[predict_df.index, enc_col]

# ==========================================
# BƯỚC 4 — CHUẨN BỊ X, Y VÀ KIỂM TRA SẠCH RÁC NULL
# ==========================================
# Vì các cột số đã ở sẵn dạng int/float, chỉ cần kiểm tra điền khuyết bằng Median
for col in feature_cols:
    if train_df[col].isna().sum() > 0 or predict_df[col].isna().sum() > 0:
        fill_value = train_df[col].median()
        train_df[col] = train_df[col].fillna(fill_value)
        predict_df[col] = predict_df[col].fillna(fill_value)

# Trích xuất chính xác các tập mảng X, y
X_train = train_df[feature_cols].copy()
y_train = np.log1p(train_df["Price"].astype(float))  # Giảm độ lệch phải của phân phối giá
X_pred = predict_df[feature_cols].copy()

# Chốt chặn kiểm định tuyệt đối: Sạch null và đúng kiểu số
assert (
    X_train.isna().sum().sum() == 0
), "Lỗi nghiêm trọng: X_train vẫn dính Null!"
assert (
    X_pred.isna().sum().sum() == 0
), "Lỗi nghiêm trọng: X_pred vẫn dính Null!"
print("-> Hoàn tất số hóa! Toàn bộ features đầu vào đã chuyển về số int/float.")

# ==========================================
# BƯỚC 5 — CROSS-VALIDATION ĐỂ CHỌN HYPERPARAMETER
# ==========================================
print("\n--- BƯỚC 5: ĐANG CHẠY 5-FOLD CROSS VALIDATION ---")

models = {
    "RandomForest": RandomForestRegressor(
        n_estimators=300, max_depth=12, random_state=42
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42
    ),
}

best_model_name = None
best_r2_mean = -np.inf
selected_model = None

# Đánh giá hiệu năng dựa trên thang điểm R2
for name, model in models.items():
    cv_scores = cross_val_score(
        model, X_train, y_train, cv=5, scoring="r2", n_jobs=-1
    )
    mean_score = cv_scores.mean()
    std_score = cv_scores.std()
    print(f"Model {name:<16} | R² trung bình: {mean_score:.4f} (±{std_score:.4f})")

    if mean_score > best_r2_mean:
        best_r2_mean = mean_score
        best_model_name = name
        selected_model = model

print(f"==> KẾT LUẬN: Chọn mô hình [{best_model_name}] để dự đoán chính thức.")

# ==========================================
# BƯỚC 6 — TRAIN MODEL TRÊN TOÀN BỘ TRAIN SET
# ==========================================
print(f"\n--- BƯỚC 6: TIẾN HÀNH TRAIN MÔ HÌNH {best_model_name.upper()} ---")
selected_model.fit(X_train, y_train)

# ==========================================
# BƯỚC 7 — PREDICT VÀ INVERSE TRANSFORM
# ==========================================
predicted_log = selected_model.predict(X_pred)
predicted_price = np.expm1(predicted_log)  # Trả ngược log về giá tiền VND gốc

# ==========================================
# BƯỚC 8 — GÁN GIÁ TRỊ VÀO DATASET
# ==========================================
predict_df["Price"] = predicted_price
df.loc[predict_df.index, "Price"] = predicted_price

# ==========================================
# BƯỚC 9 — VALIDATE KẾT QUẢ VÀ GHIM BIÊN (CLAMP)
# ==========================================
print("\n--- BƯỚC 9: VALIDATE KẾT QUẢ VÀ SỬA ĐỔI OUTLIER ---")

# 1. Kiểm tra sót Null
final_null_count = df["Price"].isna().sum()
print(f"1. Số lượng dòng trống (Null) còn sót lại ở cột Price: {final_null_count}")

# 2. Thiết lập ranh giới giá trị thực tế tại thị trường Việt Nam
MIN_PRICE = 300_000
MAX_PRICE = 70_000_000

under_min = (df["Price"] < MIN_PRICE).sum()
over_max = (df["Price"] > MAX_PRICE).sum()
print(f"2. Phát hiện bất thường: {under_min} máy giá < 300k, {over_max} máy giá > 70tr.")

# Ép biên tự động bằng hàm clip
df["Price"] = df["Price"].clip(lower=MIN_PRICE, upper=MAX_PRICE)
print("-> Đã xử lý ép biên (Clamp) các máy có giá trị dự đoán dị biệt về vùng an toàn.")

# 3. Trích xuất ngẫu nhiên 12 dòng vừa dự đoán để kiểm tra logic kinh tế
print("\n3. SPOT-CHECK ĐỐI CHIẾU LOGIC NGẪU NHIÊN 12 DÒNG VỪA ĐIỀN GIÁ:")
columns_to_show = ["Brand", "RAM", "ROM", "antutu_11", "Price"]
spot_check_data = df.loc[predict_df.index].sample(n=12, random_state=42)
print(spot_check_data[columns_to_show])

-> Đã vá 83 dòng Chipset trống dựa trên Name và Gen.
Kích thước tập Train (Có giá): 295 dòng
Kích thước tập Predict (Cần điền giá): 300 dòng
-> Hoàn tất số hóa! Toàn bộ features đầu vào đã chuyển về số int/float.

--- BƯỚC 5: ĐANG CHẠY 5-FOLD CROSS VALIDATION ---
Model RandomForest     | R² trung bình: 0.8653 (±0.0399)
Model GradientBoosting | R² trung bình: 0.8802 (±0.0469)
==> KẾT LUẬN: Chọn mô hình [GradientBoosting] để dự đoán chính thức.

--- BƯỚC 6: TIẾN HÀNH TRAIN MÔ HÌNH GRADIENTBOOSTING ---

--- BƯỚC 9: VALIDATE KẾT QUẢ VÀ SỬA ĐỔI OUTLIER ---
1. Số lượng dòng trống (Null) còn sót lại ở cột Price: 0
2. Phát hiện bất thường: 0 máy giá < 300k, 0 máy giá > 70tr.
-> Đã xử lý ép biên (Clamp) các máy có giá trị dự đoán dị biệt về vùng an toàn.

3. SPOT-CHECK ĐỐI CHIẾU LOGIC NGẪU NHIÊN 12 DÒNG VỪA ĐIỀN GIÁ:
       Brand   RAM    ROM  antutu_11         Price
203   realme   8.0  128.0   419670.0  6.349575e+06
266   xiaomi   8.0  128.0   419670.0  8.140421e+06
152    honor   8.0  256.0  

In [23]:
duplicate_count = df.duplicated(subset=["Name"]).sum()
print(f"Số lượng dòng bị trùng tên thiết bị: {duplicate_count}")

Số lượng dòng bị trùng tên thiết bị: 150


In [24]:
df= df.groupby('Name').agg(
    **{col: (col, 'first') for col in df.columns if col not in ['Name', 'RAM', 'ROM']},
    RAM_min=('RAM', 'min'),
    RAM_max=('RAM', 'max'),
    ROM_min=('ROM', 'min'),
    ROM_max=('ROM', 'max'),
).reset_index()

In [25]:
df[df['Refresh Rate'] > 144.0][['Name', 'Refresh Rate']]

,Name,Refresh Rate
375,xiaomi 17 pro,2160.0
376,xiaomi 17 pro max,2160.0


In [26]:
df.loc[df['Name'].str.contains('xiaomi 17', case=False, na=False), 'Refresh Rate'] = 120.0

In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 445 entries, 0 to 444
Data columns (total 45 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Name              445 non-null    str    
 1   Screen Size       445 non-null    float64
 2   Display           445 non-null    str    
 3   Chipset           445 non-null    str    
 4   NFC               445 non-null    int64  
 5   Battery           445 non-null    float64
 6   Price             445 non-null    float64
 7   Brand             445 non-null    str    
 8   OS_Name           445 non-null    str    
 9   OS_Version        445 non-null    float64
 10  antutu_11         445 non-null    float64
 11  clock             445 non-null    float64
 12  gpu               445 non-null    str    
 13  Chipset_name      445 non-null    str    
 14  Chipset_gen       445 non-null    str    
 15  gpu_name          445 non-null    str    
 16  gpu_gen           445 non-null    str    
 17  Refresh 

In [28]:
drop_cols = ['gpu', 'Chipset_enc', 'Chipset', 'Chipset_name_enc', 'Chipset_gen_enc', 'gpu_name_enc', 'gpu_gen_enc', 'Brand_enc', 'core_null_count']

df.drop(columns = drop_cols, inplace=True)

In [31]:
df['Chipset_name'].value_counts()

Chipset_name
Snapdragon    177
MediaTek      114
Other          64
Exynos         33
Apple          25
Unisoc         23
Kirin           9
Name: count, dtype: int64

In [30]:
# df.to_csv('preprocess_1_new.csv')